# Day 34: Technical journal and Phase 2 publication

**Phase 2: EDA & Feature Engineering**

## Objective
Technical journal and Phase 2 publication

## Task
Write technical blog post on interesting Phase 2 findings (pressing, networks, or clustering). Publish on website. Update progress tracker.

**Deliverable:** Published technical blog post + updated tracker.

---

### Instructions:
*Treat this as your course workbook. Write your code, notes, or execution steps below.*

Remember: 
- Document your decisions.
- Use clear variable names.
- If today's task involves creating a script in `src/`, a dashboard in `apps/`, or documentation in `docs/`, you can use this notebook to test your logic or simply write your reflections and proofs of execution (e.g. running terminal commands with `!`).



## 1. Phase 2 Audit, Website Architecture & Narrative Thread

**Objective:** Audit existing Phase 2 notebooks (Days 19, 24, 27), inspect `docs/` website assets and `docs/_data/progress.yml`, and lock in the 3-act storytelling structure for our Phase 2 Technical Journal.

**Editorial Rationale:** Rather than treating Pressing, Passing Networks, and Player Clustering as isolated exercises, uniting them into a single **World Cup 2022 Tactical Story** (*Without the Ball $\rightarrow$ With the Ball $\rightarrow$ The Individual Archetypes*) creates a cohesive narrative for both tactical analysts and technical recruiters.

**Hypothesis:** Exporting three high-resolution, publication-ready figures into `reports/figures/` and `docs/images/` and publishing a dedicated HTML case study (`docs/phase2-journal.html`) will elevate our Phase 2 work into a flagship portfolio piece.

In [1]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
REPO_ROOT = cwd if (cwd / "data").exists() else cwd.parent.parent
FIG_DIR = REPO_ROOT / "reports" / "figures"
DOCS_IMG_DIR = REPO_ROOT / "docs" / "images"
PROGRESS_PATH = REPO_ROOT / "docs" / "_data" / "progress.yml"

print("Existing figures in reports/figures/:")
for p in sorted(FIG_DIR.glob("*")):
    if p.name != ".gitkeep":
        print(f" - {p.name} ({p.stat().st_size / 1024:.1f} KB)")

print(f"\nWebsite index exists: {(REPO_ROOT / 'docs' / 'index.html').exists()}")
print(f"Reference project page exists: {(REPO_ROOT / 'docs' / 'footballscout.html').exists()}")
print(f"Progress tracker exists: {PROGRESS_PATH.exists()}")


Existing figures in reports/figures/:
 - head_to_head_lottin.png (194.8 KB)
 - head_to_head_modric.png (183.6 KB)
 - head_to_head_modrić.png (183.6 KB)
 - messi_vs_mbappe_final_evolution.gif (924.8 KB)
 - nfl_combine_correlation_heatmap.png (445.6 KB)
 - nfl_combine_scout_fallacy.png (885.0 KB)
 - phase2_act1_pressing_ppda.png (227.2 KB)
 - phase2_act2_passing_network.png (273.1 KB)
 - player_similarity_lottin.png (189.5 KB)
 - player_similarity_mbappe.png (190.0 KB)
 - player_similarity_modric.png (178.4 KB)
 - player_similarity_modrić.png (178.4 KB)
 - tactical_universe_2d.png (861.2 KB)
 - wc2022_territory_control.gif (767.8 KB)

Website index exists: True
Reference project page exists: True
Progress tracker exists: True


### Editorial Decisions: 3-Act Storytelling Arc

- **Narrative Thread (From Macro to Micro):**
  1. **Act I — Pressing & PPDA (Day 24):** *Disrupting the opponent without the ball* (PPDA vs. High Turnovers across Qatar 2022).
  2. **Act II — Passing Networks (Day 19):** *Organizing possession with the ball* (Graph centrality, hubs, and spatial passing structure).
  3. **Act III — Player Clustering (Day 27):** *Mapping individual roles inside the system* (GMM + PCA + HDBSCAN tactical archetypes & "Unicorns", anchored by `tactical_universe_2d.png`).
- **Publication Format:** Standalone editorial page `docs/phase2-journal.html` harmonized with `docs/assets/css/style.css` and linked from `docs/index.html`, plus full Phase 2 completion update in `docs/_data/progress.yml` and `docs/assets/js/app.js`.

## 2. Curation & High-Resolution Export of Editorial Figures

**Objective:** Generate and export two publication-grade figures (220 DPI, dark-mode tactical aesthetic) to `reports/figures/` and `docs/images/`:
1. **`phase2_act1_pressing_ppda.png`:** Tournament-wide Pressing Quadrant (*PPDA* vs. *High Turnovers per Match*, $x > 60$) across all 32 World Cup 2022 teams.
2. **`phase2_act2_passing_network.png`:** Argentina's Passing Network in the 2022 World Cup Final (`match_id = 3869685`) prior to the first substitution, encoding pass volume (edge width) and Betweenness Centrality (node color).

**Editorial Rationale:** Consistent typography, dark-canvas contrast (`#0e1117`), and explicit StatsBomb attribution make the figures immediately legible inside `docs/phase2-journal.html` without visual clutter.

In [2]:
import shutil
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from mplsoccer import Pitch

EVENTS_PATH = REPO_ROOT / "data" / "raw" / "statsbomb" / "sb_events_43_106.parquet"
df_events = pd.read_parquet(EVENTS_PATH)
df_reg = df_events[df_events["period"].isin([1, 2, 3, 4])].copy()
df_reg["x"] = df_reg["location"].str[0]
df_reg["y"] = df_reg["location"].str[1]

# --- FIGURE 1: Act I — 32-Team Pressing Quadrant (PPDA vs High Turnovers / Match) ---
match_teams = df_reg[["match_id", "team"]].drop_duplicates()
opp_map = match_teams.merge(match_teams, on="match_id", suffixes=("", "_opp"))
opp_map = opp_map[opp_map["team"] != opp_map["team_opp"]].rename(columns={"team_opp": "opp_team"})
df_reg = df_reg.merge(opp_map, on=["match_id", "team"], how="left")

passes_comp = df_reg[(df_reg["type"] == "Pass") & (df_reg["pass_outcome"].isna())]
opp_buildup = passes_comp[passes_comp["x"] < 72].groupby("opp_team").size().rename("opp_buildup_passes")
def_acts = df_reg[df_reg["type"].isin({"Pressure", "Interception", "Duel", "Foul Committed"}) & (df_reg["x"] > 48)].groupby("team").size().rename("def_actions")
high_turnovers = df_reg[df_reg["type"].isin({"Ball Recovery", "Interception"}) & (df_reg["x"] > 60)].groupby("team").size().rename("high_turnovers")
matches_played = df_reg.groupby("team")["match_id"].nunique().rename("matches")

press_df = pd.concat([matches_played, opp_buildup, def_acts, high_turnovers], axis=1).fillna(0).reset_index(names="team")
press_df["ppda"] = press_df["opp_buildup_passes"] / press_df["def_actions"]
press_df["high_turnovers_pg"] = press_df["high_turnovers"] / press_df["matches"]

fig, ax = plt.subplots(figsize=(11, 7), facecolor="#0e1117")
ax.set_facecolor("#0e1117")
med_ppda = press_df["ppda"].median()
med_ht = press_df["high_turnovers_pg"].median()
ax.axvline(med_ppda, color="#444c56", ls="--", lw=1.2)
ax.axhline(med_ht, color="#444c56", ls="--", lw=1.2)

sc = ax.scatter(
    press_df["ppda"], press_df["high_turnovers_pg"],
    s=press_df["matches"] * 48, c=press_df["high_turnovers_pg"] / press_df["ppda"],
    cmap="viridis", alpha=0.9, edgecolors="white", linewidth=0.9
)
labels_act1 = {"Spain", "Germany", "Argentina", "France", "Brazil", "England", "Morocco", "Croatia", "Japan", "Senegal", "Costa Rica", "Poland"}
for _, r in press_df.iterrows():
    if r["team"] in labels_act1:
        ax.annotate(r["team"], (r["ppda"], r["high_turnovers_pg"]), xytext=(5, 4), textcoords="offset points", color="#e6edf3", fontsize=8.5, fontweight="bold")

ax.invert_xaxis()  # Lower PPDA = more intense pressing on the right
ax.text(press_df["ppda"].min() + 0.05, press_df["high_turnovers_pg"].max() - 0.3, "ELITE HIGH PRESS\n(Low PPDA + High Turnovers)", color="#3fb950", fontsize=9, fontweight="bold", ha="right")
ax.text(press_df["ppda"].max() - 0.05, press_df["high_turnovers_pg"].min() + 0.3, "LOW-BLOCK / PASSIVE\n(High PPDA + Few Turnovers)", color="#8b949e", fontsize=9, fontweight="bold", ha="left")
ax.set_title("Act I: World Cup 2022 Pressing Intensity (PPDA) vs. High Turnovers", color="white", fontsize=13.5, fontweight="bold", pad=12)
ax.set_xlabel("PPDA (Inverted: Right = More Intense Pressing)", color="#c9d1d9")
ax.set_ylabel("High Turnovers per Match (x > 60)", color="#c9d1d9")
ax.tick_params(colors="#c9d1d9")
for spine in ax.spines.values():
    spine.set_color("#30363d")
ax.grid(color="#21262d", linestyle=":", alpha=0.7)
fig.text(0.99, 0.01, "Source: StatsBomb Open Data (WC 2022) | Bubble size = Matches played", color="#8b949e", fontsize=8, ha="right")

act1_path = FIG_DIR / "phase2_act1_pressing_ppda.png"
fig.savefig(act1_path, dpi=220, facecolor="#0e1117", bbox_inches="tight")
shutil.copy2(act1_path, DOCS_IMG_DIR / act1_path.name)
plt.close(fig)

# --- FIGURE 2: Act II — Argentina Passing Network (WC 2022 Final, Pre-First-Sub) ---
FINAL_ID = 3869685
arg_events = df_events[(df_events["match_id"] == FINAL_ID) & (df_events["team"] == "Argentina")].copy()
first_sub_idx = arg_events[arg_events["type"] == "Substitution"]["index"].min()
arg_pre = arg_events[arg_events["index"] < first_sub_idx].copy()
arg_passes = arg_pre[(arg_pre["type"] == "Pass") & (arg_pre["pass_outcome"].isna()) & arg_pre["pass_recipient"].notna()].copy()
arg_passes["x"] = arg_passes["location"].str[0]
arg_passes["y"] = arg_passes["location"].str[1]
arg_passes["x_end"] = arg_passes["pass_end_location"].str[0]
arg_passes["y_end"] = arg_passes["pass_end_location"].str[1]

avg_locs = arg_passes.groupby("player").agg(x=("x", "mean"), y=("y", "mean"), count=("x", "size")).reset_index()
pair_passes = arg_passes.groupby(["player", "pass_recipient"]).size().rename("pass_count").reset_index()

G = nx.DiGraph()
for _, r in pair_passes.iterrows():
    G.add_edge(r["player"], r["pass_recipient"], weight=r["pass_count"])
betw = nx.betweenness_centrality(G, weight=None)
avg_locs["betweenness"] = avg_locs["player"].map(betw).fillna(0)

pitch = Pitch(pitch_type="statsbomb", pitch_color="#0e1117", line_color="#444c56", line_zorder=1)
fig2, ax2 = plt.subplots(figsize=(11, 7), facecolor="#0e1117")
pitch.draw(ax=ax2)

loc_dict = avg_locs.set_index("player")[["x", "y"]].to_dict("index")
for _, r in pair_passes[pair_passes["pass_count"] >= 3].iterrows():
    if r["player"] in loc_dict and r["pass_recipient"] in loc_dict:
        p1, p2 = loc_dict[r["player"]], loc_dict[r["pass_recipient"]]
        ax2.plot([p1["x"], p2["x"]], [p1["y"], p2["y"]], color="#58a6ff", lw=r["pass_count"] * 0.55, alpha=0.45, zorder=2)

nodes = pitch.scatter(
    avg_locs["x"], avg_locs["y"], s=avg_locs["count"] * 22,
    c=avg_locs["betweenness"], cmap="plasma", edgecolors="white", linewidth=1.6, ax=ax2, zorder=3
)
for _, r in avg_locs.iterrows():
    last_name = r["player"].split()[-1]
    ax2.text(r["x"], r["y"] - 3.8, last_name, color="white", fontsize=8.5, fontweight="bold", ha="center", va="top", zorder=4)

cbar2 = fig2.colorbar(nodes, ax=ax2, fraction=0.03, pad=0.02)
cbar2.set_label("Betweenness Centrality", color="#c9d1d9")
cbar2.ax.yaxis.set_tick_params(color="#c9d1d9")
plt.setp(plt.getp(cbar2.ax.axes, "yticklabels"), color="#c9d1d9")
ax2.set_title("Act II: Argentina Passing Network — WC 2022 Final (Pre-Sub, Min 0–41)", color="white", fontsize=13.5, fontweight="bold", pad=12)
fig2.text(0.98, 0.02, "Source: StatsBomb Open Data | Node size = Passes, Color = Betweenness Centrality, Edges >= 3 passes", color="#8b949e", fontsize=8, ha="right")

act2_path = FIG_DIR / "phase2_act2_passing_network.png"
fig2.savefig(act2_path, dpi=220, facecolor="#0e1117", bbox_inches="tight")
shutil.copy2(act2_path, DOCS_IMG_DIR / act2_path.name)
plt.close(fig2)

print("Top 5 Pressing Teams (lowest PPDA):")
print(press_df.sort_values("ppda")[["team", "matches", "ppda", "high_turnovers_pg"]].head(5).round(2).to_string(index=False))
print("\nArgentina Top 5 Hubs by Betweenness Centrality:")
print(avg_locs.sort_values("betweenness", ascending=False)[["player", "count", "betweenness"]].head(5).round(3).to_string(index=False))
print(f"\n[EXPORTED] {act1_path.name} & {act2_path.name} to reports/figures/ and docs/images/")


Top 5 Pressing Teams (lowest PPDA):
   team  matches  ppda  high_turnovers_pg
  Spain        4  1.65              24.25
 Mexico        3  1.98              29.00
Germany        3  2.14              34.00
Ecuador        3  2.16              25.33
England        5  2.18              20.80

Argentina Top 5 Hubs by Betweenness Centrality:
                         player  count  betweenness
                 Enzo Fernandez     40        0.081
        Nicolás Hernán Otamendi     44        0.070
Ángel Fabián Di María Hernández     25        0.055
         Rodrigo Javier De Paul     37        0.041
        Cristian Gabriel Romero     39        0.029

[EXPORTED] phase2_act1_pressing_ppda.png & phase2_act2_passing_network.png to reports/figures/ and docs/images/


### Editorial Decisions: Visual Curation Summary

- **Unified dark-mode visual language:** All three hero figures (`phase2_act1_pressing_ppda.png`, `phase2_act2_passing_network.png`, and `tactical_universe_2d.png`) share a consistent `#0e1117` canvas and StatsBomb Open Data attribution.
- **Narrative continuity:** Act I quantifies pressing efficiency across all 32 teams; Act II zooms into Argentina's Final passing graph (`Betweenness Centrality`) to show how possession flows once recovered; Act III zooms into the individual player archetypes.

## 3. Website Publication & Progress Tracker Verification

**Objective:** Verify that the standalone Phase 2 Technical Journal (`docs/phase2-journal.html`), its navigation/card links in `docs/index.html`, all three high-resolution figures in `docs/images/`, and the progress tracker (`docs/_data/progress.yml` + `docs/assets/js/app.js`) are synchronized and ready for deployment.

**Quality Checks:**
- Confirm zero broken image references inside `docs/phase2-journal.html`.
- Confirm Days 1 through 34 are marked `completed` in `docs/_data/progress.yml`.
- Confirm editorial typography hygiene (zero em-dashes).

In [3]:
import re

journal_path = REPO_ROOT / "docs" / "phase2-journal.html"
index_path = REPO_ROOT / "docs" / "index.html"
journal_html = journal_path.read_text(encoding="utf-8")
index_html = index_path.read_text(encoding="utf-8")
progress_txt = PROGRESS_PATH.read_text(encoding="utf-8")

# 1. Check images referenced in phase2-journal.html
expected_imgs = [
    "phase2_act1_pressing_ppda.png",
    "phase2_act2_passing_network.png",
    "tactical_universe_2d.png",
]
for img in expected_imgs:
    exists = (DOCS_IMG_DIR / img).exists()
    referenced = img in journal_html
    print(f"Image {img}: exists={exists}, referenced={referenced}")

# 2. Check link from index.html and em-dash hygiene
print(f"Linked from docs/index.html: {'phase2-journal.html' in index_html}")
print(f"Em-dashes in phase2-journal.html: {journal_html.count(chr(8212))}")

# 3. Count completed days in progress.yml
completed_count = len(re.findall(r'status:\s*"completed"', progress_txt))
print(f"Completed days in progress.yml: {completed_count} / 100")


Image phase2_act1_pressing_ppda.png: exists=True, referenced=True
Image phase2_act2_passing_network.png: exists=True, referenced=True
Image tactical_universe_2d.png: exists=True, referenced=True
Linked from docs/index.html: True
Em-dashes in phase2-journal.html: 0
Completed days in progress.yml: 34 / 100


### Key Takeaways: Phase 2 Publication Complete

- **Cohesive 3-Act Storytelling:** Published `docs/phase2-journal.html` uniting Pressing/PPDA (Act I), Passing Networks (Act II), and Unsupervised Player Clustering (Act III) with custom 220 DPI visuals and honest methodological limitations.
- **Tracker & Portfolio Synchronized:** Linked the article from `docs/index.html` and updated `docs/_data/progress.yml` and `docs/assets/js/app.js` to reflect **34/100 days completed**.